# Formula-SWCT Generalization on CUB-200

This notebook evaluates the final Formula-SWCT method on the CUB-200 fine-grained bird-classification dataset using seeds 42, 43, and 44.

CUB-200 is not one of the datasets used to define the Formula-SWCT calibration targets.

The evaluation compares:

- Baseline frozen ResNet-18,
- original Single-Parameter Curvature Tuning (S-CT),
- Formula-SWCT.

Formula-SWCT follows the predict-first, train-once protocol:

1. Fit the Formula-SWCT predictor using only the calibration datasets.
2. Collect stage statistics from the CUB-200 training split.
3. Predict four stage-specific beta values before downstream training.
4. Freeze the predicted beta values.
5. Extract fixed train, validation, and test features once.
6. Train one downstream linear classifier.
7. Select the classifier checkpoint using validation accuracy.
8. Evaluate once on the test split.

No CUB-200-specific beta sweep or beta optimization is performed by Formula-SWCT.

## Environment Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The repository is cloned, the `stage-wise-ct` branch is selected, and the required research dependencies are installed.

After dependency installation, restart the runtime before continuing.

In [ ]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

### Runtime Restart Required

Restart the Colab runtime after dependency installation.

After restarting, do not rerun the installation cell. Continue from the next cell.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

## CUB-200 Dataset Setup

The CUB-200 train, validation, and test loaders are constructed using the project's standard transfer-learning pipeline.

This cell verifies the dataset sizes and tensor structure before running the evaluation methods.

In [ ]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_cub200",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

# Baseline and Original S-CT Reference

The Baseline and original Single-Parameter Curvature Tuning methods are evaluated on CUB-200 using seeds 42, 43, and 44.

The Baseline uses the frozen ImageNet-pretrained ResNet-18 with the original ReLU activations.

S-CT performs its original validation-based search over a single global beta value shared across the network.

These experiments provide the reference against which Formula-SWCT is evaluated.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-28 10:47:09.511 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed42.log
2026-09-28 10:47:09.514 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 10:48:09.003 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 10:48:09.023 | INFO     | __main__:main:88 - Number of trainable parameters: 102600
2026-09-28 10:48:09.023 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 10:49:00.248 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.639931, Accuracy: 3.12%
2026-09-28 10:49:00.525 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.213926, Accuracy: 2.07%
2026-09-28 10:49:00.810 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.119506, Val Accuracy: 13.19%
2026-09-28 10:49:00.811 | INFO     | train:linear_probe:190 - New best validation accuracy: 13.19 at epoch 1
2026-09-28 10:49:00.858 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.969887, Accuracy: 

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
2026-09-28 11:40:55.788 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed43.log
2026-09-28 11:40:55.791 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 11:41:55.971 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 11:41:55.986 | INFO     | __main__:main:88 - Number of trainable parameters: 102600
2026-09-28 11:41:55.986 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 11:42:48.526 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.576430, Accuracy: 0.00%
2026-09-28 11:42:48.806 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.249439, Accuracy: 1.64%
2026-09-28 11:42:49.095 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.096121, Val Accuracy: 16.86%
2026-09-28 11:42:49.096 | INFO     | train:linear_probe:190 - New best validation accuracy: 16.86 at epoch 1
2026-09-28 11:42:49.150 | INFO     | train:trai

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
2026-09-28 12:20:06.897 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed44.log
2026-09-28 12:20:06.901 | INFO     | __main__:main:58 - Running on cuda
2026-09-28 12:21:08.243 | INFO     | __main__:main:85 - Testing baseline...
2026-09-28 12:21:08.258 | INFO     | __main__:main:88 - Number of trainable parameters: 102600
2026-09-28 12:21:08.258 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-28 12:22:01.301 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.421100, Accuracy: 0.00%
2026-09-28 12:22:01.594 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.247695, Accuracy: 2.04%
2026-09-28 12:22:01.881 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.169811, Val Accuracy: 15.69%
2026-09-28 12:22:01.882 | INFO     | train:linear_probe:190 - New best validation accuracy: 15.69 at epoch 1
2026-09-28 12:22:01.936 | INFO     | train:trai

# Fit the Frozen Formula-SWCT Predictor

Because this notebook is intended to run independently in a fresh Colab runtime, the Formula-SWCT predictor is reconstructed before evaluating CUB-200.

The formula is fitted using only the three development/calibration datasets:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each calibration dataset and seed, five descriptors are measured from the frozen ImageNet-pretrained ResNet-18:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

A ridge-regression predictor with lambda = 0.10 is fitted from these calibration observations.

CUB-200 is not used to modify or retune the formula.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: FIT DIRECT BETA FORMULA
No beta sweep and no downstream training are performed here.

data/train-00000-of-00001.parquet: 100% 144M/144M [00:02<00:00, 56.7MB/s] 
data/validation-00000-of-00001.parquet: 100% 18.5M/18.5M [00:01<00:00, 12.4MB/s]
data/test-00000-of-00001.parquet: 100% 17.7M/17.7M [00:01<00:00, 12.9MB/s]
Generating train split: 1034 examples [00:00, 1677.26 examples/s]
Generating validation split: 133 examples [00:00, 3514.50 examples/s]
Generating test split: 128 examples [00:00, 3352.68 examples/s]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 182MB/s]



100% 625M/625M [12:09<00:00, 857kB/s]  



100% 345M/345M [06:29<00:00, 886kB/s]  
100% 502/502 [00:00<00:00, 1.52MB/s]
100% 15.0k/15.0k [00:00<00:00, 31.9MB/s]



FITTED FORMULA
beta_hat = clip(b0 + sum(w_j * z_j), 0.7, 0.99)
intercept:

# Formula-SWCT Evaluation on CUB-200

The frozen Formula-SWCT predictor is now evaluated using seeds 42, 43, and 44.

For each seed:

- CUB-200 training-split stage statistics are collected,
- four stage-specific beta values are predicted before downstream training,
- the predicted beta values are frozen,
- features are extracted once,
- one downstream classifier is trained.

The same fitted Formula-SWCT predictor is used for all three seeds.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: cub200
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.
README.md: 2.44kB [00:00, 12.8MB/s]
cub200.py: 7.90kB [00:00, 23.6MB/s]
Generating train split: 5994 examples [06:51, 14.58 examples/s]
Generating test split: 5794 examples [06:34, 14.67 examples/s]

Predicted betas BEFORE training:
Stage 1: 0.917763
Stage 2: 0.903602
Stage 3: 0.906269
Stage 4: 0.918496
Predicted beta vector: [0.917763, 0.903602, 0.906269, 0.918496]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.917763, 0.903602, 0.906269, 0.918496]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=4.093281 | val_acc=15.28%
Epoch 02 | val_loss=2.761331 | val_acc=43.66%
Epoch 03 | val_loss=2.257832 | val_acc=50.00%
Epoch 04 | val_loss=1.987194 | val_acc=52.84%
Epoch 05 | val_loss=1.829629 | val_acc=56.01%
Epoch 06 | val_loss=1.715508 | val_acc=5

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: cub200
Seed: 43
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.917333
Stage 2: 0.903422
Stage 3: 0.906188
Stage 4: 0.918051
Predicted beta vector: [0.917333, 0.903422, 0.906188, 0.918051]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.917333, 0.903422, 0.906189, 0.918051]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=4.062046 | val_acc=15.28%
Epoch 02 | val_loss=2.767533 | val_acc=40.90%
Epoch 03 | val_loss=2.195104 | val_acc=51.09%
Epoch 04 | val_loss=1.937005 | val_acc=54.26%
Epoch 05 | val_loss=1.752334 | val_acc=59.27%
Epoch 06 | val_loss=1.647553 | val_acc=59.35%
Epoch 07 | val_loss=1.594385 | val_acc=59.10%
Epoch 08 | val_loss=1.553577 | val_acc=58.76%
Epoch 09 | val_loss=1.501110 | val_acc=61.02%
Epoch 10 | val_loss=1.488079 | val_acc=60.10%
Epoch 11 

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: cub200
Seed: 44
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.917849
Stage 2: 0.904017
Stage 3: 0.906782
Stage 4: 0.919171
Predicted beta vector: [0.917849, 0.904017, 0.906782, 0.919171]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.917849, 0.904017, 0.906782, 0.919171]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=4.110297 | val_acc=15.78%
Epoch 02 | val_loss=2.879867 | val_acc=35.73%
Epoch 03 | val_loss=2.319130 | val_acc=47.58%
Epoch 04 | val_loss=2.068305 | val_acc=50.75%
Epoch 05 | val_loss=1.897725 | val_acc=53.67%
Epoch 06 | val_loss=1.794616 | val_acc=55.76%
Epoch 07 | val_loss=1.755507 | val_acc=55.34%
Epoch 08 | val_loss=1.707788 | val_acc=56.51%
Epoch 09 | val_loss=1.672388 | val_acc=56.34%
Epoch 10 | val_loss=1.632304 | val_acc=57.18%
Epoch 11 

## Previously Completed Formula-SWCT Runs

The three CUB-200 Formula-SWCT experiments were previously completed successfully.

### Seed 42

Predicted beta vector:

**[0.917763, 0.903602, 0.906269, 0.918496]**

- Best validation accuracy: **61.94%**
- Best validation epoch: **14**
- Test accuracy: **63.72%**

### Seed 43

Predicted beta vector:

**[0.917333, 0.903422, 0.906188, 0.918051]**

- Best validation accuracy: **64.77%**
- Best validation epoch: **18**
- Test accuracy: **63.13%**

### Seed 44

Predicted beta vector:

**[0.917849, 0.904017, 0.906782, 0.919171]**

- Best validation accuracy: **59.60%**
- Best validation epoch: **15**
- Test accuracy: **62.62%**

The close agreement between predicted beta vectors across seeds indicates that the pre-training stage statistics used by Formula-SWCT are highly stable for CUB-200.

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

baseline_accs = []
sct_accs = []
sct_betas = []

formula_accs = []
formula_val_accs = []
formula_betas = []

for seed in seeds:

    # Baseline + S-CT
    with open(
        f"results/base_imagenet_to_cub200_resnet18_seed{seed}.json",
        "r"
    ) as f:
        baseline = json.load(f)

    with open(
        f"results/ct_imagenet_to_cub200_resnet18_seed{seed}.json",
        "r"
    ) as f:
        sct = json.load(f)

    baseline_accs.append(baseline["accuracy"])
    sct_accs.append(sct["accuracy"])
    sct_betas.append(sct["beta"])

    # Formula-SWCT
    with open(
        f"results/formula_swct_imagenet_to_cub200_resnet18_seed{seed}.json",
        "r"
    ) as f:
        formula = json.load(f)

    formula_accs.append(formula["test_acc"])
    formula_val_accs.append(formula["best_val_acc"])
    formula_betas.append(formula["predicted_betas"])


baseline_accs = np.array(baseline_accs)
sct_accs = np.array(sct_accs)
formula_accs = np.array(formula_accs)
formula_betas = np.array(formula_betas)


print("=" * 70)
print("CUB-200 RESULTS — RESNET-18")
print("=" * 70)

print("\nBASELINE")
for seed, acc in zip(seeds, baseline_accs):
    print(f"Seed {seed}: {acc:.2f}%")

print(
    f"Mean: {baseline_accs.mean():.2f}% "
    f"± {baseline_accs.std(ddof=1):.2f}"
)


print("\nS-CT")
for seed, acc, beta in zip(seeds, sct_accs, sct_betas):
    print(
        f"Seed {seed}: "
        f"test={acc:.2f}% | beta={beta:.2f}"
    )

print(
    f"Mean: {sct_accs.mean():.2f}% "
    f"± {sct_accs.std(ddof=1):.2f}"
)


print("\nFORMULA-SWCT")
for seed, acc, val, betas in zip(
    seeds,
    formula_accs,
    formula_val_accs,
    formula_betas
):
    print(
        f"Seed {seed}: "
        f"test={acc:.2f}% | "
        f"best val={val:.2f}% | "
        f"betas={[round(x, 6) for x in betas]}"
    )

print(
    f"\nMean: {formula_accs.mean():.2f}% "
    f"± {formula_accs.std(ddof=1):.2f}"
)

print(
    "\nMean predicted betas:",
    np.round(formula_betas.mean(axis=0), 6)
)

print("\nMETHOD DIFFERENCES")
print(
    f"Formula-SWCT - Baseline: "
    f"{formula_accs.mean() - baseline_accs.mean():+.2f} pp"
)
print(
    f"Formula-SWCT - S-CT: "
    f"{formula_accs.mean() - sct_accs.mean():+.2f} pp"
)

# CUB-200 Results Summary

Across seeds 42, 43, and 44:

| Method | Test Accuracy |
|---|---:|
| Baseline | **62.85 ± 0.35%** |
| S-CT | **62.66 ± 0.35%** |
| Formula-SWCT | **63.16 ± 0.55%** |

Formula-SWCT therefore improves mean CUB-200 test accuracy by approximately:

- **+0.31 percentage points** relative to the Baseline,
- **+0.50 percentage points** relative to S-CT.

The predicted Formula-SWCT beta vectors are highly consistent across the three random seeds and remain near approximately 0.90–0.92 for all four ResNet-18 stages.

Unlike S-CT, Formula-SWCT obtains these four curvature parameters without a target-specific beta sweep or beta optimization.